## Codi gen DATA_5 Psychiatric_diagnosis

Distribució de la Mostra (N=1000)
D'acord amb les dades de la cohorte:
- Controles (Sense diagnòstic): 66,7% ($\approx 667$ pacients)
- Casos (Amb diagnòstic): 33,3% ($\approx 333$ pacients)

In [ ]:
import pandas as pd
import numpy as np

# 1. Carreguem la teva taula base (suposem que ja tens els 1000 IDs)
path_pacients='C:\Users\laiar\Desktop\TFG\data_sintetica\Data-sintetica\csv\1_Sociodemographic\pacients_100k_sintetic.csv'
df_5 = pd.read_csv(path_pacients)

# 2. Paràmetres de diagnòstics (Prevalença en la Cohorte de Casos) [cite: 249, 250, 283]
diagnostics_prevalencia = {
    'Anxiety Disorders (F40-F48)': 0.3138,
    'Mood Disorders (F30-F39)': 0.1883,
    'Substance Use Disorders': 0.1359,  # Usem el valor del 13.59% 
    'Schizophrenia & Psychotic Disorders (F20-F29)': 0.035
}

def generar_diagnostics_psiquiatrics(df):
    df_psic = df.copy()
    np.random.seed(42)
    n_total = len(df_psic)

    indexs_casos = df_psic[df_psic['Grup_Salut_Mental'] == 'Cas'].index.tolist()
    
    for diag, prev in diagnostics_prevalencia.items():
        # La prevalença s'aplica SOBRE el nombre de casos (333), no sobre el total [cite: 249]
        n_a_assignar = int(round(prev * n_casos))
        
        # Triem pacients que encara no tinguin aquest diagnòstic
        candidats = [idx for idx in indexs_casos if df_psic.at[idx, diag] == 'No']
        
        if candidats:
            triats = np.random.choice(candidats, size=min(n_a_assignar, len(candidats)), replace=False)
            df_psic.loc[triats, diag] = 'Sí'
            
    return df_psic

# Execució
df_diagnostics = generar_diagnostics_psiquiatrics(df_final)

## Psychiatric diseases with code ICD-10:
| Catalan / Spanish | English Translation | ICD-10 Range | Prevalence (Applied) |
| :--- | :--- | :--- | :--- |
| **Trastorns de l'Ansietat** | **Anxiety Disorders** | F40–F48 | 31.38% |
| **Trastorns de l'Ànim / de l'Humor** | **Mood (Affective) Disorders** | F30–F39 | 18.83% |
| **Esquizofrènia i T. Psicòtics** | **Schizophrenia and Psychotic Disorders** | F20–F29 | 3.50% |
| **Trastorns per ús de substàncies** | **Substance Use Disorders** | F10–F19 | 10.89% |

In [ ]:
import pandas as pd
import numpy as np
import random

# 1. Definició de categories amb els seus rangs numèrics
config_malalties = {
    'Trastorns Ansietat': {'min': 40, 'max': 48, 'prev': 0.3138},
    'Trastorns Anim': {'min': 30, 'max': 39, 'prev': 0.1883},
    'Esquizofrenia_Psicotics': {'min': 20, 'max': 29, 'prev': 0.035},
    'Trastorns Substancies': {'min': 10, 'max': 19, 'prev': 0.1089}
}

def generar_codi_aleatori(categoria):
    conf = config_malalties[categoria]
    # Triem un número aleatori entre el mínim i el màxim del rang
    num = random.randint(conf['min'], conf['max'])
    return f"F{num}"

def generar_dataset_psiquiatric(n_total=100000):
    np.random.seed(42)
    random.seed(42) # Seed per a la part de random pur
    
    df = pd.DataFrame({'id_pacient': range(1, n_total + 1)})
    malalties = list(config_malalties.keys())
    
    # Inicialitzem columnes Sí/No
    for m in malalties:
        df[m] = 'No'
    
    diag_nom_list = []
    diag_codi_list = []

    for i in range(n_total):
        actius = []
        for m in malalties:
            if np.random.rand() < config_malalties[m]['prev']:
                actius.append(m)
        
        # Garantim que no hi hagi pacients control
        if not actius:
            principal = random.choice(malalties)
            actius = [principal]
        else:
            principal = actius[0]
            
        # Marquem els 'Sí'
        df.loc[i, actius] = 'Sí'
        
        # Assignem nom i un CODI ALEATORI dins del rang
        diag_nom_list.append(principal)
        diag_codi_list.append(generar_codi_aleatori(principal))
    
    df['nom_diagnostic'] = diag_nom_list
    df['codi_cie10'] = diag_codi_list
    
    return df

# 2. Execució i mostra de resultats
df_final = generar_dataset_psiquiatric(100000)

# Comprovem que els codis varien (ex: dins d'Ansietat tindrem F40, F45, F48...)
print("Exemple de variabilitat de codis per categoria:")
print(df_final.groupby('nom_diagnostic')['codi_cie10'].unique().apply(lambda x: sorted(x[:5])))

df_final.head()

Exemple de variabilitat de codis per categoria:
nom_diagnostic
Esquizofrenia_Psicotics    [F21, F22, F23, F26, F29]
Trastorns Anim             [F30, F32, F34, F35, F38]
Trastorns Ansietat         [F40, F41, F43, F44, F46]
Trastorns Substancies      [F11, F13, F14, F15, F16]
Name: codi_cie10, dtype: object


,id_pacient,Trastorns Ansietat,Trastorns Anim,Esquizofrenia_Psicotics,Trastorns Substancies,nom_diagnostic,codi_cie10
0,1,Sí,No,No,No,Trastorns Ansietat,F40
1,2,Sí,Sí,No,No,Trastorns Ansietat,F44
2,3,No,No,Sí,No,Esquizofrenia_Psicotics,F23
3,4,No,Sí,No,No,Trastorns Anim,F32
4,5,Sí,No,No,No,Trastorns Ansietat,F41
